In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Setup paths
base_path = Path("data/fruits-360")
splits = ["Training", "Validation", "Test"]

data = []

for split in splits:
    split_path = base_path / split
    if not split_path.exists():
        continue
        
    categories = [d for d in split_path.iterdir() if d.is_dir()]
    
    for category_path in categories:
        category_name = category_path.name
        file_count = len(list(category_path.glob("*.jpg")))
        data.append({
            "Split": split,
            "Category": category_name,
            "Count": file_count
        })

df = pd.DataFrame(data)

# Summary Statistics
if df.empty or "Split" not in df.columns:
    print("No data found or 'Split' column missing. Please check the base_path and data availability.")
else:
    stats = df.groupby("Split")["Count"].agg(["sum", "mean", "min", "max", "count"]).rename(columns={"count": "Classes", "sum": "Total Images"})
    print("Dataset Overview:")
    print(stats)

    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # 1. Total Images per Split
    df.groupby("Split")["Count"].sum().plot(kind="bar", ax=axes[0], color=['skyblue', 'salmon', 'lightgreen'])
    axes[0].set_title("Total Images per Split")
    axes[0].set_ylabel("Count")

    # 2. Distribution of images per class (Training split)
    train_df = df[df["Split"] == "Training"].sort_values("Count", ascending=False)
    axes[1].hist(train_df["Count"], bins=20, color='purple', alpha=0.7)
    axes[1].set_title("Distribution of Images per Class (Training)")
    axes[1].set_xlabel("Images per Class")
    axes[1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

    # Print top/bottom classes to check for imbalance
    print("\nTop 5 Classes (Training):")
    print(train_df.head(5)[["Category", "Count"]].to_string(index=False))

    print("\nBottom 5 Classes (Training):")
    print(train_df.tail(5)[["Category", "Count"]].to_string(index=False))

No data found or 'Split' column missing. Please check the base_path and data availability.


In [2]:
import os
import pandas as pd
from pathlib import Path

base_path = Path("data/fruits-360")
splits = ["Training", "Validation", "Test"]
# Support multiple image extensions just in case
extensions = {".jpg", ".jpeg", ".png", ".bmp"}

data = []

for split in splits:
    split_path = base_path / split
    if not split_path.exists():
        continue
    
    # Get all subdirectories (categories)
    categories = sorted([d for d in split_path.iterdir() if d.is_dir()])
    
    for cat_path in categories:
        # Count files by checking if suffix is in our extensions set
        files = [f for f in cat_path.iterdir() if f.suffix.lower() in extensions]
        data.append({
            "Category": cat_path.name,
            "Split": split,
            "Count": len(files)
        })

df = pd.DataFrame(data)

# Check if data is available before proceeding
if df.empty or "Category" not in df.columns:
    print("No data found or 'Category' column missing. Please check the base_path and data availability.")
else:
    # Create a pivot table for an alphabetical overview of all categories
    pivot_df = df.pivot(index="Category", columns="Split", values="Count").fillna(0).astype(int)
    pivot_df = pivot_df.sort_index() # Alphabetical sort

    # Add a total column for the class
    pivot_df["Total_Per_Class"] = pivot_df.sum(axis=1)

    print(f"Total Images in Dataset: {pivot_df['Total_Per_Class'].sum()}")
    print(f"Unique Categories: {len(pivot_df)}")
    print("-" * 30)

    # Display the full table (Jupyter will provide a scrollable view)
    pd.set_option('display.max_rows', None)
    display(pivot_df)

No data found or 'Category' column missing. Please check the base_path and data availability.


In [3]:
# find Unit_0/data/fruits-360 -type f \( -iname "*.jpg" -o -iname "*.jpeg" -o -iname "*.png" \) | wc -l
# Call in terminal e.g.

In [4]:
import pandas as pd

# Create pivot_df from df (assuming df is defined from a previous cell)
if df.empty or "Category" not in df.columns:
    print("No data found or 'Category' column missing. Please check the base_path and data availability.")
else:
    pivot_df = df.pivot(index="Category", columns="Split", values="Count").fillna(0).astype(int)
    pivot_df = pivot_df.sort_index()  # Alphabetical sort
    # Add a total column for the class
    pivot_df["Total_Per_Class"] = pivot_df.sum(axis=1)

    # 1. Create mapping: Use first word, handle both spaces and underscores, normalize case
    pivot_df['Super_Category'] = pivot_df.index.str.split(r'[ _]').str[0].str.capitalize()

    # 2. Aggregate counts by the new Super-Category
    super_cat_stats = pivot_df.groupby('Super_Category').agg({
        'Training': 'sum',
        'Validation': 'sum',
        'Test': 'sum',
        'Total_Per_Class': 'sum'
    })

    # 3. Add a count of how many sub-variants (original folders) belong to each Super-Category
    super_cat_stats['Sub_Variant_Count'] = pivot_df.groupby('Super_Category').size()

    # 4. Sort by total volume
    super_cat_stats = super_cat_stats.sort_values("Total_Per_Class", ascending=False)

    # Display results
    print(f"Total Super-Categories: {len(super_cat_stats)}")
    print("-" * 30)
    display(super_cat_stats)

    # Summary of the top 5 largest groups
    print("\nTop 5 Super-Categories by Image Volume:")
    print(super_cat_stats.head(5).to_string())

No data found or 'Category' column missing. Please check the base_path and data availability.


In [5]:
# Create pivot_df from df (assuming df is defined from a previous cell)
if df.empty or "Category" not in df.columns:
    print("No data found or 'Category' column missing. Please check the base_path and data availability.")
else:
    pivot_df = df.pivot(index="Category", columns="Split", values="Count").fillna(0).astype(int)
    pivot_df = pivot_df.sort_index()  # Alphabetical sort
    # Add a total column for the class
    pivot_df["Total_Per_Class"] = pivot_df.sum(axis=1)

    # 1. Create mapping: Use first word, handle both spaces and underscores, normalize case
    pivot_df['Super_Category'] = pivot_df.index.str.split(r'[ _]').str[0].str.capitalize()

    # 2. Aggregate counts by the new Super-Category
    super_cat_stats = pivot_df.groupby('Super_Category').agg({
        'Training': 'sum',
        'Validation': 'sum',
        'Test': 'sum',
        'Total_Per_Class': 'sum'
    })

    # 3. Add a count of how many sub-variants (original folders) belong to each Super-Category
    super_cat_stats['Sub_Variant_Count'] = pivot_df.groupby('Super_Category').size()

    # 4. Sort by total volume
    super_cat_stats = super_cat_stats.sort_values("Total_Per_Class", ascending=False)

    # Group by Super_Category and collect all original index names (sub-variants)
    sub_variant_lists = pivot_df.groupby('Super_Category').apply(
        lambda x: ", ".join(sorted(x.index.tolist()))
    ).rename("Sub_Variants")

    # Join this list back to your super_cat_stats dataframe
    detailed_super_cat = super_cat_stats.join(sub_variant_lists)

    # Display the full table
    pd.set_option('display.max_colwidth', None)  # Ensure we see all names
    display(detailed_super_cat)

    # Specific check for 'Apple' to verify your 31 variants
    print("\nDetailed breakdown for 'Apple':")
    apple_info = detailed_super_cat.loc[['Apple']]
    print(f"Total Folders: {apple_info['Sub_Variant_Count'].values[0]}")
    print(f"Variants: {apple_info['Sub_Variants'].values[0]}")

No data found or 'Category' column missing. Please check the base_path and data availability.


In [6]:
import matplotlib.pyplot as plt
import PIL.Image as Image
import random
from pathlib import Path

def inspect_category(category_name, base_path="data/fruits-360"):
    base = Path(base_path)
    splits = ["Training", "Validation", "Test"]
    
    # 1. Print Folder Statistics
    print(f"Inspection for Category: {category_name}")
    print("-" * 30)
    
    split_files = {}
    for s in splits:
        p = base / s / category_name
        files = list(p.glob("*.jpg")) + list(p.glob("*.png"))
        split_files[s] = files
        print(f"{s.ljust(12)}: {len(files)} images")
    
    # 2. Prepare Sampling (Prioritizing Training for display)
    all_images = split_files["Training"] if split_files["Training"] else split_files["Test"]
    
    if len(all_images) < 9:
        print(f"\nWarning: Not enough images to show 3x3 grid (found {len(all_images)})")
        return

    sample_paths = random.sample(all_images, 9)
    
    # 3. Plot 3x3 Grid
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    fig.suptitle(f"Random Samples from {category_name} (Training)", fontsize=16)
    
    for i, ax in enumerate(axes.flat):
        img = Image.open(sample_paths[i])
        ax.imshow(img)
        ax.set_title(f"Size: {img.size}")
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

In [7]:
# Usage
inspect_category("Apple 10")

Inspection for Category: Apple 10
------------------------------
Training    : 0 images
Validation  : 0 images
Test        : 0 images



In [8]:
# Usage
inspect_category("Papaya 2")

Inspection for Category: Papaya 2
------------------------------
Training    : 0 images
Validation  : 0 images
Test        : 0 images

